# Dallas 311 Intelligence — Multi-Threshold GPU Training
**500,000 rows · Thresholds: 24h, 48h, 72h, 96h · cuML & XGBoost**

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── Cell 2: Install RAPIDS & XGBoost ────────────────────────────────────────
!pip install -q xgboost scikit-learn pandas numpy joblib

In [ ]:
# ── Cell 3: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT  = '/content/drive/MyDrive/Dallas311'
DATA_PATH   = f'{DRIVE_ROOT}/dallas_311_500k.csv'
ARTIFACT_DIR = f'{DRIVE_ROOT}/artifacts_multi_threshold'
os.makedirs(ARTIFACT_DIR, exist_ok=True)

In [ ]:
# ── Cell 4: Load and Clean ──────────────────────────────────────────────────
import pandas as pd
import numpy as np

df = pd.read_csv(DATA_PATH, low_memory=False)

# Clean & Filter
df.drop(columns=['Service Request Number', 'Unique Key', 'Address'], errors='ignore', inplace=True)
df['Created Date'] = pd.to_datetime(df['Created Date'], format='%Y %b %d %I:%M:%S %p', errors='coerce')
df['Closed Date']  = pd.to_datetime(df['Closed Date'],  format='%Y %b %d %I:%M:%S %p', errors='coerce')

# Calculate response time
df['days_to_close'] = (df['Closed Date'] - df['Created Date']).dt.total_seconds() / 3600
df = df[df['days_to_close'] > 0].copy()

# Time features
df['hour']        = df['Created Date'].dt.hour
df['day_of_week'] = df['Created Date'].dt.dayofweek
df['month']       = df['Created Date'].dt.month
print(f'Base dataset prepared: {len(df):,} rows.')

In [ ]:
# ── Cell 5: One-Hot Encoding & Preprocessing ────────────────────────────────
from sklearn.preprocessing import LabelEncoder
import joblib

LEAKAGE = ['Closed Date', 'Update Date', 'Status', 'Outcome', 
           'Lat_Long Location', 'Overall Service Request Due Date', 'Created Date']
df.drop(columns=[c for c in LEAKAGE if c in df.columns], inplace=True)

if 'Estimated Response Time Description' in df.columns:
    df['ERT_days'] = df['Estimated Response Time Description'].str.extract(r'(\d+)').astype(float)
    df.drop(columns=['Estimated Response Time Description'], inplace=True)

# Cast to str
for col in df.select_dtypes(include=['object', 'category']).columns:
    df[col] = df[col].astype(str)

if 'Service Request Type' in df.columns:
    top_15 = df['Service Request Type'].value_counts().nlargest(15).index
    df['Service Request Type'] = df['Service Request Type'].where(df['Service Request Type'].isin(top_15), 'Other')

# Label Encode
encoders = {}
for col in ['Priority', 'Method Received Description', 'Department']:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        encoders[col] = le

# OHE
cat_cols = [c for c in df.select_dtypes(include=['object', 'category']).columns if c != 'days_to_close']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

X_full = df.drop(columns=['days_to_close'])
y_days = df['days_to_close']
feature_names = list(X_full.columns)

joblib.dump(feature_names, f'{ARTIFACT_DIR}/feature_names.pkl')
print(f'Ready for loop. {len(feature_names)} features.')

In [ ]:
# ── Cell 6: Multi-Threshold Training Loop ───────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import joblib
import json

thresholds = [24, 48, 72, 96]
overall_metrics = {}

for hr in thresholds:
    print(f'\n======================================================')
    print(f'🎯 TRAINING THRESHOLD: {hr} HOURS')
    print(f'======================================================')
    
    # Generate dynamic binary target
    y = (y_days <= hr).astype(int)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_full, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Impute & Scale
    imp = SimpleImputer(strategy='median')
    X_tr = pd.DataFrame(imp.fit_transform(X_train), columns=feature_names)
    X_te = pd.DataFrame(imp.transform(X_test),      columns=feature_names)
    
    scaler = StandardScaler()
    X_tr = pd.DataFrame(scaler.fit_transform(X_tr), columns=feature_names)
    X_te = pd.DataFrame(scaler.transform(X_te),      columns=feature_names)
    
    # Save preprocessors for this specific hour
    joblib.dump(imp,    f'{ARTIFACT_DIR}/imputer_{hr}h.pkl')
    joblib.dump(scaler, f'{ARTIFACT_DIR}/scaler_{hr}h.pkl')

    threshold_results = {}

    # --- 1. Logistic Regression (cuML) ---
    print('-> Training Logistic Regression on GPU...')
    lr = LogisticRegression(max_iter=1000, n_jobs=-1)
    lr.fit(X_tr.astype('float32'), y_train.astype('float32'))
    joblib.dump(lr, f'{ARTIFACT_DIR}/logistic_regression_{hr}h.joblib')
    
    # --- 2. Random Forest (cuML) ---
    print('-> Training Random Forest on GPU...')
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(X_tr.astype('float32'), y_train.astype('int32'))
    joblib.dump(rf, f'{ARTIFACT_DIR}/random_forest_{hr}h.joblib')
    
    # --- 3. XGBoost ---
    print('-> Training XGBoost on GPU...')
    xgb = XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        reg_alpha=0.1, reg_lambda=2.0, min_child_weight=10,
        device='cuda', tree_method='hist', random_state=42, verbosity=0
    )
    xgb.fit(X_tr, y_train)
    xgb.save_model(f'{ARTIFACT_DIR}/xgb_model_{hr}h.json')
    
    # --- Evaluate All ---
    from sklearn.metrics import roc_auc_score
    
    for name, model in [('LR', lr), ('RF', rf), ('XGB', xgb)]:
        test_data = X_te.astype('float32') if name != 'XGB' else X_te
        probs = model.predict_proba(test_data)
        
        if hasattr(probs, 'iloc'):
            y_prob = probs.iloc[:, 1].to_numpy()
        elif False: # hasattr(probs, 'get'): disabled for sklearn
            y_prob = probs.get()[:, 1]
        else:
            y_prob = probs[:, 1]
            
        auc = float(roc_auc_score(y_test, y_prob))
        threshold_results[name] = round(auc, 4)
        print(f'[{name}] ROC-AUC: {auc:.4f}')
        
    overall_metrics[f'{hr}h'] = threshold_results

with open(f'{ARTIFACT_DIR}/multi_threshold_summary.json', 'w') as f:
    json.dump(overall_metrics, f, indent=2)

print('\n=== All Multi-Threshold Training Passes Complete ===')